# DINOv3 LoRA + BiGRU-GatedABMIL Training

This notebook trains the main model: DINOv3 with LoRA adapters, scan-level BiGRU-GatedABMIL loss, and optional Seg-CQ500 slice-level auxiliary loss.

Run `notebooks/01_vast_frozen_baseline.ipynb` first through data preparation and Seg-CQ500 slice label generation.

In [ ]:
from pathlib import Path
import json
import torch

assert Path('configs/vast/lora_mil.yaml').exists(), 'Run from repo root.'
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Required prepared files

- `data/raw/hf/dinov3-vitb16-pretrain-lvd1689m`
- `data/processed/dicom_index_fast.csv`
- `data/processed/labels_matched.csv`
- `splits/cq500_seed42.csv`
- optional but recommended: `data/processed/slice_labels.csv` from Seg-CQ500

In [ ]:
for path in [
    'data/raw/hf/dinov3-vitb16-pretrain-lvd1689m',
    'data/processed/dicom_index_fast.csv',
    'data/processed/labels_matched.csv',
    'splits/cq500_seed42.csv',
]:
    print(path, Path(path).exists())
print('slice labels', Path('data/processed/slice_labels.csv').exists())

## Train

If you hit CUDA OOM, reduce `training.max_slices` or `training.slice_batch_size` in `configs/vast/lora_mil.yaml`.

In [ ]:
!python scripts/train_lora_mil.py --config configs/vast/lora_mil.yaml

In [ ]:
metrics_path = Path('data/models/lora_bigru_abmil/metrics.json')
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))